In [ ]:
from flask import Flask, render_template, request
import requests
from bs4 import BeautifulSoup
import re
import csv
import os

app = Flask(__name__)

CSV_FILE = "emails.csv"

# Function to scrape emails
def scrape_emails(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        text = soup.get_text()

        emails = set(re.findall(
            r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
            text
        ))

        return emails

    except Exception as e:
        print("Error:", e)
        return set()


# Function to save emails to CSV
def save_to_csv(url, emails):

    file_exists = os.path.isfile(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(["URL", "Email"])

        for email in emails:
            writer.writerow([url, email])


# Homepage
@app.route("/", methods=["GET", "POST"])
def index():

    emails = []
    url = ""

    if request.method == "POST":

        url = request.form.get("url")

        if url:
            emails = scrape_emails(url)

            if emails:
                save_to_csv(url, emails)

    return render_template("index.html", emails=emails, url=url)


if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [07/Mar/2026 20:01:37] "GET / HTTP/1.1" 200 -
